# Phase 3 - graph

Building out the LangGraph orchestration loop for Phase 3. Starts with item 4's skeleton (`docs/build-order.md`) and evolves through the rest of Phase 3 as later items land - one notebook, not a new one per item.

Section order: state/reducers, then node functions, then routing functions, then the graph itself.

## Imports & setup

Requires the MCP server running separately first: `uv run python -c "from app.mcp_server.server import mcp; import uvicorn; uvicorn.run(mcp.http_app(), host='127.0.0.1', port=8001)"`

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import uuid
import asyncio
from datetime import datetime
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from langchain_mcp_adapters.client import MultiServerMCPClient

from app.config import MODEL, get_secret, GCP_PROJECT_ID
from app.telemetry.writer import build_telemetry_row, write_telemetry_row
from app.orchestrator.context import get_static_context

os.environ["ANTHROPIC_API_KEY"] = get_secret("anthropic-api-key", GCP_PROJECT_ID)

MAX_ANSWER_CHARS = 6000
MAX_LENGTH_RETRIES = 2
MAX_VERIFY_RETRIES = 2

SYSTEM_MESSAGE = SystemMessage(content=get_static_context())

mcp_client = MultiServerMCPClient({
    "analytics": {"transport": "streamable_http", "url": "http://127.0.0.1:8001/mcp"},
})
_tools = await mcp_client.get_tools()
MCP_TOOLS = {t.name: t for t in _tools}


def check_answer_length(answer_markdown: str) -> bool:
    return len(answer_markdown) <= MAX_ANSWER_CHARS


print("MCP tools loaded:", list(MCP_TOOLS))

## State & reducers

In [ ]:
class ToolCallRecord(TypedDict):
    id: str
    name: str
    args: dict
    result: list[dict] | str
    success: bool
    error: str | None
    started_at: datetime
    completed_at: datetime


class TurnError(TypedDict):
    stage: str
    error_type: str
    message: str
    occurred_at: datetime


def append_list(existing: list, new: list) -> list:
    """Accumulates a list across graph supersteps."""
    return existing + new


class AgentState(TypedDict):
    # Set once, at invocation
    question: str
    conversation_id: str
    user_id: str
    turn_started_at: datetime
    filter_context: list[dict]
    active_page: str | None
    image_base64: str | None

    # Accumulated during the loop
    messages: Annotated[list[BaseMessage], add_messages]
    tool_calls: Annotated[list[ToolCallRecord], append_list]
    iteration_count: int

    # Separate retry budgets
    verification_retry_count: int
    length_retry_count: int

    verified: bool

    # Resource accumulators
    bytes_consumed: int
    prompt_tokens: int
    completion_tokens: int
    llm_calls: int

    errors: Annotated[list[TurnError], append_list]
    cancelled: bool

    # Guardrail outcomes
    needs_approval: bool
    pending_queries: list[dict]
    deferred_dax: list[dict]
    estimated_cost: str | None
    cost_cap_exceeded: bool
    iteration_cap_hit: bool

    # Building toward AgentResponse
    answer_markdown: str
    chart_url: str | None
    sources: list[str]


print("AgentState fields:", list(AgentState.__annotations__))
print("Total field count:", len(AgentState.__annotations__))

## Node functions

Item 4's scope (`docs/build-order.md`) using the real `ping` tool already on the MCP server instead of an empty list, so `call_tool` gets exercised for real too. `agent` writes `answer_markdown` itself whenever it has no tool calls - no separate structured-output call. `verify` is a placeholder; real pooled-matching verification is item 6.

In [ ]:
async def agent_node(state: AgentState) -> dict:
    """Calls the LLM with tools bound. Emits tool calls, or writes answer_markdown."""
    model = ChatAnthropic(model=MODEL).bind_tools(list(MCP_TOOLS.values()))
    response = await model.ainvoke([SYSTEM_MESSAGE, *state["messages"]])
    update = {"messages": [response], "llm_calls": state["llm_calls"] + 1}
    if not response.tool_calls:
        update["answer_markdown"] = response.content
    return update


async def call_tool_node(state: AgentState) -> dict:
    """Dispatches every tool call in the batch and returns their ToolMessages."""
    tool_calls = state["messages"][-1].tool_calls
    results = await asyncio.gather(
        *[MCP_TOOLS[tc["name"]].ainvoke(tc) for tc in tool_calls])
    return {"messages": results, "iteration_count": state["iteration_count"] + 1}


async def check_length_node(state: AgentState) -> dict:
    """Checks answer_markdown length; on failure, requests a shorter answer."""
    if check_answer_length(state["answer_markdown"]):
        return {}
    return {
        "length_retry_count": state["length_retry_count"] + 1,
        "messages": [HumanMessage(content=(
            "Your answer is too long to display. Summarize the key findings "
            "concisely, or generate a chart instead of listing rows."))],
    }


async def verify_node(state: AgentState) -> dict:
    """Placeholder -- always verifies. Real checks land later."""
    return {"verified": True}


async def finalize_node(state: AgentState) -> dict:
    """Builds and writes the telemetry row for this turn."""
    row = build_telemetry_row(
        conversation_id=state["conversation_id"],
        user_id=state["user_id"],
        question=state["question"],
        answer_markdown=state["answer_markdown"],
        turn_started_at=state["turn_started_at"],
        turn_completed_at=datetime.now(),
    )
    await write_telemetry_row(row)
    return {}


print("5 nodes defined")

## Routing functions

In [ ]:
def route_after_agent(state: AgentState) -> str:
    return "call_tool" if state["messages"][-1].tool_calls else "check_length"


def route_after_call_tool(state: AgentState) -> str:
    if state["cancelled"] or state["needs_approval"] or state["cost_cap_exceeded"]:
        return "finalize"
    return "agent"   # also the iteration_cap_hit path


def route_after_check_length(state: AgentState) -> str:
    if check_answer_length(state["answer_markdown"]):
        return "verify"
    if state["length_retry_count"] < MAX_LENGTH_RETRIES:
        return "agent"
    return "finalize"


def route_after_verify(state: AgentState) -> str:
    if state["verified"]:
        return "finalize"
    if state["verification_retry_count"] < MAX_VERIFY_RETRIES:
        return "agent"
    return "finalize"

## Graph

`route_entry`/`execute_approved` aren't built until item 11, so this graph starts at `agent` directly.

In [ ]:
g = StateGraph(AgentState)
g.add_node("agent",        agent_node)
g.add_node("call_tool",    call_tool_node)
g.add_node("check_length", check_length_node)
g.add_node("verify",       verify_node)
g.add_node("finalize",     finalize_node)

g.add_edge(START, "agent")

# path_map (2nd dict arg) is what makes get_graph() draw the real edges.
g.add_conditional_edges("agent", route_after_agent,
    {"call_tool": "call_tool", "check_length": "check_length"})
g.add_conditional_edges("call_tool", route_after_call_tool,
    {"finalize": "finalize", "agent": "agent"})
g.add_conditional_edges("check_length", route_after_check_length,
    {"verify": "verify", "agent": "agent", "finalize": "finalize"})
g.add_conditional_edges("verify", route_after_verify,
    {"finalize": "finalize", "agent": "agent"})

g.add_edge("finalize", END)
graph = g.compile()

print("graph compiled:", list(graph.get_graph().nodes))

## Run it end to end

Real Claude calls, real telemetry write - via `astream`, not `ainvoke`, matching the gateway's actual invocation pattern from the start (`.claude/rules/gateway.md`). With only `ping` bound, a simple question goes `agent -> check_length -> verify -> finalize` on the first pass.

In [ ]:
initial_state = {
    "question": "In one sentence, what is the Instacart dataset used for?",
    "conversation_id": str(uuid.uuid4()),
    "user_id": "notebook-test-user",
    "turn_started_at": datetime.now(),
    "filter_context": [],
    "active_page": None,
    "image_base64": None,
    "messages": [HumanMessage(content="In one sentence, what is the Instacart dataset used for?")],
    "tool_calls": [],
    "iteration_count": 0,
    "verification_retry_count": 0,
    "length_retry_count": 0,
    "verified": False,
    "bytes_consumed": 0,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "llm_calls": 0,
    "errors": [],
    "cancelled": False,
    "needs_approval": False,
    "pending_queries": [],
    "deferred_dax": [],
    "estimated_cost": None,
    "cost_cap_exceeded": False,
    "iteration_cap_hit": False,
    "answer_markdown": "",
    "chart_url": None,
    "sources": [],
}

final_state = None
async for chunk in graph.astream(initial_state, stream_mode=["updates", "values"]):
    kind, data = chunk
    if kind == "values":
        final_state = data
        continue
    node = next(iter(data))
    print(f"-> node ran: {node}")

print()
print("answer_markdown:", final_state["answer_markdown"])
print("verified:", final_state["verified"])
print("llm_calls:", final_state["llm_calls"])
print("message count:", len(final_state["messages"]))
assert final_state["verified"] is True
assert len(final_state["answer_markdown"]) > 0
print()
print("PASS -- full loop ran agent -> check_length -> verify -> finalize, telemetry written")